# Simple CNN testing

In [13]:
import torch
import wandb
import time
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from src.cnn_utils import get_dataset

In [2]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")

print(device)

cuda


In [3]:
train_dataset, val_dataset = get_dataset()

In [4]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


### Create Training loop for models

In [ ]:
def train_eval(model, optimizer, nepochs, batch_size, training_data, validation_data, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN',run_name=None, use_wandb=True):
    """
    Train and evaluate a model.
    Logs train/validation loss and accuracy to Weights & Biases if use_wandb=True.
    """
    cost_hist = []
    cost_hist_test = []
    acc_hist = []
    acc_hist_test = []

    model = model.to(device) # <-- move model to device (GPU or CPU)
    cost_ce = torch.nn.CrossEntropyLoss().to(device)
    
    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    
    if use_wandb:
        wandb.init(
            entity=entity,
            project=project,
            name=run_name,
            settings=wandb.Settings(init_timeout=300),
            config={
                "epochs": nepochs,
                "batch_size": batch_size,
                "optimizer": optimizer.__class__.__name__,
                "loss": "CrossEntropyLoss",
                "device": str(device),
                "model": model.__class__.__name__
            }
        )
        wandb.watch(model, log="all", log_freq=100)

    for epoch in range(nepochs):
        start = time.perf_counter()

        model.train()
        size = len(train_loader.dataset)
        nbatches = len(train_loader)
        cost, acc = 0.0, 0.0
        for batch, (X, Y) in enumerate(train_loader):
            X,Y = X.to(device),Y.to(device)
            pred = model(X)
            loss = cost_ce(pred, Y)
            cost += loss.item()
            acc += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

            # gradient, parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        cost /= nbatches
        acc /= size

        model.eval()
        size_test = len(val_loader.dataset)
        nbatches_test = len(val_loader)
        cost_test, acc_test = 0.0, 0.0     

        with torch.no_grad():
            for X, Y in val_loader:
                X,Y = X.to(device),Y.to(device)
                pred = model(X)
                cost_test += cost_ce(pred, Y).item()
                acc_test += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

        cost_test /= nbatches_test
        acc_test /= size_test

        end = time.perf_counter()
        epoch_time = end - start

        print(f"Epoch {epoch}: Train cost: {round(cost, 4)}, accuracy: {round(acc, 4)}, Validation cost: {round(cost_test, 4)}, accuracy: {round(acc_test, 4)} (Time: {round(epoch_time, 4)} seconds)")

        cost_hist.append(cost)
        cost_hist_test.append(cost_test)
        acc_hist.append(acc)
        acc_hist_test.append(acc_test)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": cost,
                "train_accuracy": acc,
                "val_loss": cost_test,
                "val_accuracy": acc_test,
                "lr": optimizer.param_groups[0]['lr'],
                "epoch_time": epoch_time
            })

    if use_wandb:
        wandb.finish()

    return cost_hist, cost_hist_test, acc_hist, acc_hist_test

### Creating shallow CNN-model

In [10]:
# creat a simple model with one convolutional layer and two fully connected layers

class first_model(nn.Module):
    
    def __init__(self, units=128):
        super(first_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [11]:
# create an model and its summary

model = first_model(128) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 128]      51,380,352
              ReLU-6                  [-1, 128]               0
            Linear-7                   [-1, 10]           1,290
Total params: 51,382,538
Trainable params: 51,382,538
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 196.01
Estimated Total Size (MB): 227.21
----------------------------------------------------------------


Initiate Training

In [ ]:
batch_size = 64
nepochs = 50
lr = 0.001
units = 128

model = first_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e30_lr0.001_u128_1l_no-reg', use_wandb=True)


Epoch 0: Train cost: 2.207545274098714, accuracy: 0.20025, Validation cost: 2.1467549572599696, accuracy: 0.24983333333333332 (Time: 30.545374100000117)
Epoch 1: Train cost: 2.0994765961964923, accuracy: 0.27045833333333336, Validation cost: 2.059368625600287, accuracy: 0.30516666666666664 (Time: 31.213550000000396)
Epoch 2: Train cost: 2.0210647751490276, accuracy: 0.305875, Validation cost: 1.994484250849866, accuracy: 0.329 (Time: 30.782830099999956)
Epoch 3: Train cost: 1.953537075360616, accuracy: 0.33195833333333336, Validation cost: 1.9538064294673028, accuracy: 0.31883333333333336 (Time: 29.379701799999566)
Epoch 4: Train cost: 1.899112151145935, accuracy: 0.348, Validation cost: 1.9003696581150622, accuracy: 0.3451666666666667 (Time: 29.450798400000167)


This first model is heavily in a overfitting regime.\
Really bad vallidation accuracy and loss.\
The Result was kind of expected looking at the amount of parameters that are estimted during training (+40 Mio).\
Therefore we tried to construct a shallow model that performs better.
Possibilities to improve the model:
- heavier downsampling before passing into a fully connected layer
    --> adding more layers before fully connectde layers (Conv2d --> ReLu --> MaxPool2d)
- reduce/ make the dense layer smaller (less units)
- add regularization:
    - Dropout
    - better optimizer
    - early stopping
    - etc.


In [17]:
# creat a simple model with one convolutional layer and two fully connected layers

class improved_model(nn.Module):
    
    def __init__(self, units=128, drop=0.5):
        super(improved_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), # Conv with 32 filters, kernel size 3x3 and padding 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.Dropout(drop), # dropout rate of 0.5 as deafault value
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [19]:
model = improved_model(128, 0.5) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 128]      51,380,352
           Dropout-6                  [-1, 128]               0
              ReLU-7                  [-1, 128]               0
            Linear-8                   [-1, 10]           1,290
Total params: 51,382,538
Trainable params: 51,382,538
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 196.01
Estimated Total Size (MB): 227.21
----------------------------------------------------------------


In [ ]:
batch_size = 64
nepochs = 30
lr = 0.001
units = 128
wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter

model = first_model(units)
optimizer = torch.optim.Adam(params=model.parameters(), lr = lr, weight_decay=wd)
cost_train_adam, cost_valid_adam, acc_train_adam, acc_valid_adam = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e30_lr0.001_u128_1l_adam_drop0.5', use_wandb=True)


In [20]:
class shallow_model(nn.Module):
    
    def __init__(self, units=128):
        super(shallow_model, self).__init__()
        self.seq = nn.Sequential(
            # Layer 1---------------------------------------------------------------------------------------
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 224x224 -> 112x112

            # Layer 2---------------------------------------------------------------------------------------
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # Conv with 64 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 112x112 -> 56x56

            # output layer----------------------------------------------------------------------------------
            nn.Flatten(),
            nn.Linear(56*56*64,units),
            
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [21]:
# create an model and its summary

model = shallow_model() # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 64, 112, 112]          18,496
              ReLU-5         [-1, 64, 112, 112]               0
         MaxPool2d-6           [-1, 64, 56, 56]               0
           Flatten-7               [-1, 200704]               0
            Linear-8                  [-1, 128]      25,690,240
              ReLU-9                  [-1, 128]               0
           Linear-10                   [-1, 10]           1,290
Total params: 25,710,922
Trainable params: 25,710,922
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 42.88
Params size (MB): 98.08
Est

In [13]:
batch_size = 32
nepochs = 10
lr = 0.1
units = 100

model = simple_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='shallow_model', use_wandb=True)


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


Epoch 0: 2.255288, 0.186200, 2.141300, 0.241333
Epoch 1: 1.982461, 0.296550, 1.851362, 0.343833
Epoch 2: 1.738407, 0.388850, 1.803388, 0.367667
Epoch 3: 1.452711, 0.494950, 1.868406, 0.370333
Epoch 4: 1.101732, 0.622550, 2.119803, 0.351167
Epoch 5: 0.744286, 0.750350, 2.720517, 0.335833
Epoch 6: 0.539178, 0.829250, 3.146812, 0.337000
Epoch 7: 0.377976, 0.879450, 3.877148, 0.324333
Epoch 8: 0.288310, 0.910450, 4.103664, 0.315667
Epoch 9: 0.241948, 0.929350, 4.661720, 0.328833


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


## Hyperparameter tuning

WandB has a integrated hyperparameter sweep which we want to try out for this MPW.\
in the following section we tried to tune the model with this function